# Assignment 1: Data Preprocessing

Student: Htet Khant Linn

Date: 23 Oct 2026

Load the data first

In [1]:
import pandas as pd
import numpy as np
df=pd.read_csv('../datasets_globelEnv/Lab2_weather.csv')
df.head()

,Unnamed: 0,DateTime,StationID,PM2.5,PM10,NO2,SO2,O3,CO,Temperature,Humidity,WindSpeed,Weather,AQI,AQI_Category
0,0,2024-01-01 00:00:00,Station_C,39.967142,73.428050,NaN,17.385882,56.370175,1.258677,16.872084,65.781983,3.195104,Rainy,111.999494,Unhealthy for Sensitive
1,1,2024-01-01 01:00:00,Station_C,38.793738,62.491596,40.556577,22.743510,NaN,0.982323,15.924396,51.618483,1.276158,cloudy,109.110209,Unhealthy for Sensitive
2,2,2024-01-01 02:00:00,STATION_A,51.476885,97.711619,NaN,27.958366,37.279782,1.039817,19.935899,62.911165,8.773654,RAINY,140.340070,Unhealthy for Sensitive
3,3,2024-01-01 03:00:00,Station_D,64.372434,92.734039,50.132296,17.852464,60.408011,1.047789,20.751439,42.309779,2.553534,Cloudy,155.581130,Unhealthy
4,4,2024-01-01 04:00:00,Station_C,NaN,101.462011,57.659714,24.924136,36.624919,0.950957,17.376939,78.646998,0.968020,RAINY,136.651746,Unhealthy for Sensitive


Drop the unwanted column

In [2]:
# Drop the Unnamed: O column using df.drop

df.drop(columns='Unnamed: 0',inplace=True)
df.head()

,DateTime,StationID,PM2.5,PM10,NO2,SO2,O3,CO,Temperature,Humidity,WindSpeed,Weather,AQI,AQI_Category
0,2024-01-01 00:00:00,Station_C,39.967142,73.428050,NaN,17.385882,56.370175,1.258677,16.872084,65.781983,3.195104,Rainy,111.999494,Unhealthy for Sensitive
1,2024-01-01 01:00:00,Station_C,38.793738,62.491596,40.556577,22.743510,NaN,0.982323,15.924396,51.618483,1.276158,cloudy,109.110209,Unhealthy for Sensitive
2,2024-01-01 02:00:00,STATION_A,51.476885,97.711619,NaN,27.958366,37.279782,1.039817,19.935899,62.911165,8.773654,RAINY,140.340070,Unhealthy for Sensitive
3,2024-01-01 03:00:00,Station_D,64.372434,92.734039,50.132296,17.852464,60.408011,1.047789,20.751439,42.309779,2.553534,Cloudy,155.581130,Unhealthy
4,2024-01-01 04:00:00,Station_C,NaN,101.462011,57.659714,24.924136,36.624919,0.950957,17.376939,78.646998,0.968020,RAINY,136.651746,Unhealthy for Sensitive


# Handling Missing Values

### Q1: Sort by DateTime to ensure proper temporal imputation


Check dataset info

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2010 entries, 0 to 2009
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   DateTime      2010 non-null   str    
 1   StationID     2010 non-null   str    
 2   PM2.5         1912 non-null   float64
 3   PM10          2010 non-null   float64
 4   NO2           1909 non-null   float64
 5   SO2           2010 non-null   float64
 6   O3            1909 non-null   float64
 7   CO            2010 non-null   float64
 8   Temperature   1960 non-null   float64
 9   Humidity      2010 non-null   float64
 10  WindSpeed     2010 non-null   float64
 11  Weather       1980 non-null   str    
 12  AQI           2010 non-null   float64
 13  AQI_Category  2010 non-null   str    
dtypes: float64(10), str(4)
memory usage: 311.7 KB


Sort by DateTime: order the rows chronologically from earliest to last. Use reset_index to reset the row index aftering sorting which discard the old index.

In [4]:
df_processed = df.sort_values(by='DateTime').reset_index(drop=True)
df_processed.head()

,DateTime,StationID,PM2.5,PM10,NO2,SO2,O3,CO,Temperature,Humidity,WindSpeed,Weather,AQI,AQI_Category
0,2024-01-01 00:00:00,Station_C,39.967142,73.428050,NaN,17.385882,56.370175,1.258677,16.872084,65.781983,3.195104,Rainy,111.999494,Unhealthy for Sensitive
1,2024-01-01 01:00:00,Station_C,38.793738,62.491596,40.556577,22.743510,NaN,0.982323,15.924396,51.618483,1.276158,cloudy,109.110209,Unhealthy for Sensitive
2,2024-01-01 02:00:00,STATION_A,51.476885,97.711619,NaN,27.958366,37.279782,1.039817,19.935899,62.911165,8.773654,RAINY,140.340070,Unhealthy for Sensitive
3,2024-01-01 03:00:00,Station_D,64.372434,92.734039,50.132296,17.852464,60.408011,1.047789,20.751439,42.309779,2.553534,Cloudy,155.581130,Unhealthy
4,2024-01-01 04:00:00,Station_C,NaN,101.462011,57.659714,24.924136,36.624919,0.950957,17.376939,78.646998,0.968020,RAINY,136.651746,Unhealthy for Sensitive


# Handle pollutant missing values
### Q2 : Pollutant columns are 'PM2.5', 'NO2', 'O3' - Choose the most suitable algorithm to handle missing values and provide the reason for your choice.

In [5]:
import warnings
warnings.filterwarnings('ignore')

Group the dataset by StationID. The reason is to treat each weather station independently. Then apply .ffill() followed by .bfill(). The reason to use .bfill is to fill any remaining NaN values at the very top of the dataset.

In [6]:
pollutant_cols = ['PM2.5', 'NO2', 'O3']
for col in pollutant_cols:
    missing_before = df_processed[col].isnull().sum()
    
    df_processed[col] = df_processed.groupby('StationID')[col].ffill().bfill()
    
    missing_after = df_processed[col].isnull().sum()
    print(f"  {col}: Before : {missing_before} → After: {missing_after} missing values")

  PM2.5: Before : 98 → After: 0 missing values
  NO2: Before : 101 → After: 0 missing values
  O3: Before : 101 → After: 0 missing values


### Q3 : Handle temperature - use most suitble one that can still remain the correct pattern of that data

For temperate, interpolate approach is used. The reason is to draw a straightline between the valid temp before the gap and the valid temp after the gap. Also added ffill and bfill to check any missing value at the very first row or last row.

In [7]:
missing_before = df_processed['Temperature'].isnull().sum()
df_processed['Temperature'] = df_processed['Temperature'].interpolate(method='linear').ffill().bfill()
missing_after = df_processed['Temperature'].isnull().sum()

print(f"  Temperature: {missing_before} → {missing_after} missing values")


  Temperature: 50 → 0 missing values


### Q4: How can we handle missing values in the Weather column by filling them with the mode (most frequent value) for each hour?

In [8]:
df_processed.dtypes

DateTime            str
StationID           str
PM2.5           float64
PM10            float64
NO2             float64
SO2             float64
O3              float64
CO              float64
Temperature     float64
Humidity        float64
WindSpeed       float64
Weather             str
AQI             float64
AQI_Category        str
dtype: object

In [9]:
df_processed.info()

<class 'pandas.DataFrame'>
RangeIndex: 2010 entries, 0 to 2009
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   DateTime      2010 non-null   str    
 1   StationID     2010 non-null   str    
 2   PM2.5         2010 non-null   float64
 3   PM10          2010 non-null   float64
 4   NO2           2010 non-null   float64
 5   SO2           2010 non-null   float64
 6   O3            2010 non-null   float64
 7   CO            2010 non-null   float64
 8   Temperature   2010 non-null   float64
 9   Humidity      2010 non-null   float64
 10  WindSpeed     2010 non-null   float64
 11  Weather       1980 non-null   str    
 12  AQI           2010 non-null   float64
 13  AQI_Category  2010 non-null   str    
dtypes: float64(10), str(4)
memory usage: 312.5 KB


This convert to datatime format and Extract Hour from the date time format.

In [10]:
df_processed['DateTime']=pd.to_datetime(df_processed['DateTime'])
missing_before = df_processed['Weather'].isnull().sum()
df_processed['Hour'] = df_processed['DateTime'].dt.hour

In [11]:
df_processed.head()

,DateTime,StationID,PM2.5,PM10,NO2,SO2,O3,CO,Temperature,Humidity,WindSpeed,Weather,AQI,AQI_Category,Hour
0,2024-01-01 00:00:00,Station_C,39.967142,73.428050,40.556577,17.385882,56.370175,1.258677,16.872084,65.781983,3.195104,Rainy,111.999494,Unhealthy for Sensitive,0
1,2024-01-01 01:00:00,Station_C,38.793738,62.491596,40.556577,22.743510,56.370175,0.982323,15.924396,51.618483,1.276158,cloudy,109.110209,Unhealthy for Sensitive,1
2,2024-01-01 02:00:00,STATION_A,51.476885,97.711619,50.132296,27.958366,37.279782,1.039817,19.935899,62.911165,8.773654,RAINY,140.340070,Unhealthy for Sensitive,2
3,2024-01-01 03:00:00,Station_D,64.372434,92.734039,50.132296,17.852464,60.408011,1.047789,20.751439,42.309779,2.553534,Cloudy,155.581130,Unhealthy,3
4,2024-01-01 04:00:00,Station_C,38.793738,101.462011,57.659714,24.924136,36.624919,0.950957,17.376939,78.646998,0.968020,RAINY,136.651746,Unhealthy for Sensitive,4


Group the data by Hour and use transform and lambda function to fill with the mode value. For mode, as it return a pandas Series use [0] to selects the first value.

In [12]:
df_processed['Weather'] = df_processed.groupby('Hour')['Weather'].transform(lambda x: x.fillna(x.mode()[0]))

missing_after = df_processed['Weather'].isnull().sum()
print(f"  Weather: {missing_before} → {missing_after} missing values")


  Weather: 30 → 0 missing values


In [13]:
df_processed.isnull().sum()

DateTime        0
StationID       0
PM2.5           0
PM10            0
NO2             0
SO2             0
O3              0
CO              0
Temperature     0
Humidity        0
WindSpeed       0
Weather         0
AQI             0
AQI_Category    0
Hour            0
dtype: int64

## Datacleaning

### Q5: Remove duplicates based on DateTime and StationID

Check the rows that have exact same timestamp and StationID, set 'keep' to first to keep the first occurance.

In [14]:
print(f"\nDuplicates before cleaning: {df_processed.duplicated(subset=['DateTime', 'StationID']).sum()}")
#Fill here

df_processed.drop_duplicates(subset=['DateTime', 'StationID'], keep='first', inplace=True)

print(f"Duplicates after cleaning: {df_processed.duplicated(subset=['DateTime', 'StationID']).sum()}")



Duplicates before cleaning: 8
Duplicates after cleaning: 0


### Q6:  Standardize values of StationID (case inconsistencies)

str.upper() coverts all strings in StationID to uppercase.

In [15]:
print("\nStandardizing StationID...")
print(f"Before: {df_processed['StationID'].unique()}")

#Fill here 
df_processed['StationID'] = df_processed['StationID'].str.upper()

print(f"After: {df_processed['StationID'].unique()}")



Standardizing StationID...
Before: <ArrowStringArray>
['Station_C', 'STATION_A', 'Station_D', 'Station_B', 'station_a', 'Station_A']
Length: 6, dtype: str
After: <ArrowStringArray>
['STATION_C', 'STATION_A', 'STATION_D', 'STATION_B']
Length: 4, dtype: str


### Q6.1: Standarize Weather (case inconsistencies)

In [16]:
print("\nStandardizing Weather column...")
print(f"Before: {df_processed['Weather'].unique()}")


Standardizing Weather column...
Before: <ArrowStringArray>
['Rainy', 'cloudy', 'RAINY', 'Cloudy', 'Clear', 'clear', 'Foggy']
Length: 7, dtype: str


Use title() to converts the strings so the first letter is uppercase and the rest are lowercase

In [17]:
df_processed['Weather'] = df_processed['Weather'].str.title()
print(f"After: {df_processed['Weather'].unique()}")


After: <ArrowStringArray>
['Rainy', 'Cloudy', 'Clear', 'Foggy']
Length: 4, dtype: str


### Q7: Write Python pandas code to identify extreme outliers in pollutant columns using 3*IQR and then clip the values within the valid range.

#### In our DataFrame, we have pollutant columns (PM2.5, PM10, NO2, SO2). Show me code to:

Calculate Q1, Q3, and IQR.

Count how many outliers are present.

Print ranges before and after handling.

Clip the extreme outliers using 3IQR.

Check the values falling outside the lower bound and upper bound and sums the true values to count the outlier.

use clip to change the value above upper bound down to upper bound and raise any value below lower bound up to lower bound.

Also add remaining outliers checking process.

In [18]:
print("\nHandling outliers in pollutant measurements...")
for col in ['PM2.5', 'PM10', 'NO2', 'SO2']:
    Q1 = df_processed[col].quantile(0.25)
    Q3 = df_processed[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR  # Using 3*IQR for extreme outliers
    upper_bound = Q3 + 3 * IQR
    
    outliers = ((df_processed[col]<lower_bound) | (df_processed[col]>upper_bound)).sum()
    
    print(f"  {col}: {outliers} outliers detected")
    print(f"    Range before: [{df_processed[col].min():.2f}, {df_processed[col].max():.2f}]")
    
    # Fill here
    df_processed[col] = df_processed[col].clip(lower=lower_bound, upper=upper_bound)

    remaining_outliers = ((df_processed[col] < lower_bound) | (df_processed[col] > upper_bound)).sum()

    print(f"    Range after: [{df_processed[col].min():.2f}, {df_processed[col].max():.2f}]")
    print(f"Remaining extreme outliers: {remaining_outliers}")


Handling outliers in pollutant measurements...
  PM2.5: 15 outliers detected
    Range before: [-5.00, 337.68]
    Range after: [-5.00, 132.95]
Remaining extreme outliers: 0
  PM10: 15 outliers detected
    Range before: [-5.88, 484.87]
    Range after: [-5.88, 233.27]
Remaining extreme outliers: 0
  NO2: 0 outliers detected
    Range before: [-2.30, 69.65]
    Range after: [-2.30, 69.65]
Remaining extreme outliers: 0
  SO2: 6 outliers detected
    Range before: [15.07, 42.86]
    Range after: [15.07, 38.27]
Remaining extreme outliers: 0


### Q8: How can we extract temporal features such as Year, Month, Day, Hour, and DayOfWeek from a DateTime column in pandas?

In [19]:
df_processed.columns

Index(['DateTime', 'StationID', 'PM2.5', 'PM10', 'NO2', 'SO2', 'O3', 'CO',
       'Temperature', 'Humidity', 'WindSpeed', 'Weather', 'AQI',
       'AQI_Category', 'Hour'],
      dtype='str')

use pd.cut() to segment the numerical Hour Column into 4 discreate time of day interval. It intervals are right-inclusive by defacult (a, b], so to ensure 0 is include, right is set as False.

Note: As I have already create Hour Column in Q4, creating Hour again in Q8 will simply overwrites the Hour column with exact same values.

In [20]:
# Extract temporal features
print("\nExtracting temporal features from DateTime...")
df_processed['Year'] = df_processed['DateTime'].dt.year
df_processed['Month'] = df_processed['DateTime'].dt.month
df_processed['Day'] = df_processed['DateTime'].dt.day
df_processed['Hour'] = df_processed['DateTime'].dt.hour
df_processed['DayOfWeek'] = df_processed['DateTime'].dt.dayofweek
df_processed['IsWeekend'] = (df_processed['DayOfWeek'] >= 5).astype(int)

# Create time periods
# Fill here
df_processed['TimePeriod'] = pd.cut(df_processed['Hour'],
                                    bins=[0, 6, 12, 18, 24],
                                    labels=['Night', 'Morning', 'Afternoon', 'Evening'],
                                    right=False)

print(f"  Created features: Year, Month, Day, Hour, DayOfWeek, IsWeekend, TimePeriod")



Extracting temporal features from DateTime...
  Created features: Year, Month, Day, Hour, DayOfWeek, IsWeekend, TimePeriod


In [21]:
df_processed.head()

,DateTime,StationID,PM2.5,PM10,NO2,SO2,O3,CO,Temperature,Humidity,...,Weather,AQI,AQI_Category,Hour,Year,Month,Day,DayOfWeek,IsWeekend,TimePeriod
0,2024-01-01 00:00:00,STATION_C,39.967142,73.428050,40.556577,17.385882,56.370175,1.258677,16.872084,65.781983,...,Rainy,111.999494,Unhealthy for Sensitive,0,2024,1,1,0,0,Night
1,2024-01-01 01:00:00,STATION_C,38.793738,62.491596,40.556577,22.743510,56.370175,0.982323,15.924396,51.618483,...,Cloudy,109.110209,Unhealthy for Sensitive,1,2024,1,1,0,0,Night
2,2024-01-01 02:00:00,STATION_A,51.476885,97.711619,50.132296,27.958366,37.279782,1.039817,19.935899,62.911165,...,Rainy,140.340070,Unhealthy for Sensitive,2,2024,1,1,0,0,Night
3,2024-01-01 03:00:00,STATION_D,64.372434,92.734039,50.132296,17.852464,60.408011,1.047789,20.751439,42.309779,...,Cloudy,155.581130,Unhealthy,3,2024,1,1,0,0,Night
4,2024-01-01 04:00:00,STATION_C,38.793738,101.462011,57.659714,24.924136,36.624919,0.950957,17.376939,78.646998,...,Rainy,136.651746,Unhealthy for Sensitive,4,2024,1,1,0,0,Night


In [22]:
df_processed.tail()

,DateTime,StationID,PM2.5,PM10,NO2,SO2,O3,CO,Temperature,Humidity,...,Weather,AQI,AQI_Category,Hour,Year,Month,Day,DayOfWeek,IsWeekend,TimePeriod
2005,2024-03-24 03:00:00,STATION_A,64.843638,102.830229,25.488221,15.623744,66.530957,0.637503,29.854446,76.769431,...,Cloudy,155.824428,Unhealthy,3,2024,3,24,6,1,Night
2006,2024-03-24 04:00:00,STATION_C,57.055295,93.781225,25.346493,16.766428,67.452391,0.929768,27.824329,77.998315,...,Clear,151.803050,Unhealthy,4,2024,3,24,6,1,Night
2007,2024-03-24 05:00:00,STATION_D,50.499770,82.553480,41.203111,18.231714,60.125262,0.561717,29.666160,77.648197,...,Rainy,137.934107,Unhealthy for Sensitive,5,2024,3,24,6,1,Night
2008,2024-03-24 06:00:00,STATION_A,58.369330,93.889285,38.170936,27.887072,48.337492,1.199415,31.556305,71.028201,...,Cloudy,152.481530,Unhealthy,6,2024,3,24,6,1,Morning
2009,2024-03-24 07:00:00,STATION_C,51.869490,102.347329,36.017240,19.797610,79.031193,0.652715,23.159105,70.627467,...,Rainy,141.306785,Unhealthy for Sensitive,7,2024,3,24,6,1,Morning


### Q9: Suppose you have a PM2.5 column in a DataFrame. Write code to categorize it into:

Good (0–12)

Moderate (12.1–35.4)

Sensitive (35.5–55.4)

Unhealthy (55.5–150.4)

VeryUnhealthy (150.5–1000)

Similar to above one but for this case included_lowest = True to include the very first boundary value 0 inside the first bin instead of leaving it out as NaN.

In [23]:
print("\nCreating pollution severity categories...")

#Fill here
pm25_bins = [0, 12, 35.4, 55.4, 150.4, 1000]
pm25_labels = ['Good', 'Moderate', 'Sensitive', 'Unhealthy', 'VeryUnhealthy']

df_processed["PM2.5_Level"]= pd.cut(df_processed["PM2.5"], 
                                    bins=pm25_bins, 
                                    labels=pm25_labels,
                                    include_lowest=True
                                    )

df_processed.head()


Creating pollution severity categories...


,DateTime,StationID,PM2.5,PM10,NO2,SO2,O3,CO,Temperature,Humidity,...,AQI,AQI_Category,Hour,Year,Month,Day,DayOfWeek,IsWeekend,TimePeriod,PM2.5_Level
0,2024-01-01 00:00:00,STATION_C,39.967142,73.428050,40.556577,17.385882,56.370175,1.258677,16.872084,65.781983,...,111.999494,Unhealthy for Sensitive,0,2024,1,1,0,0,Night,Sensitive
1,2024-01-01 01:00:00,STATION_C,38.793738,62.491596,40.556577,22.743510,56.370175,0.982323,15.924396,51.618483,...,109.110209,Unhealthy for Sensitive,1,2024,1,1,0,0,Night,Sensitive
2,2024-01-01 02:00:00,STATION_A,51.476885,97.711619,50.132296,27.958366,37.279782,1.039817,19.935899,62.911165,...,140.340070,Unhealthy for Sensitive,2,2024,1,1,0,0,Night,Sensitive
3,2024-01-01 03:00:00,STATION_D,64.372434,92.734039,50.132296,17.852464,60.408011,1.047789,20.751439,42.309779,...,155.581130,Unhealthy,3,2024,1,1,0,0,Night,Unhealthy
4,2024-01-01 04:00:00,STATION_C,38.793738,101.462011,57.659714,24.924136,36.624919,0.950957,17.376939,78.646998,...,136.651746,Unhealthy for Sensitive,4,2024,1,1,0,0,Night,Sensitive


### Q10: Given an AQI_Category column with ordered categories:
['Good', 'Moderate', 'Unhealthy for Sensitive', 'Unhealthy', 'Very Unhealthy']
how can you encode them into integers while preserving the natural order?

create aqi_mapping dict and use .map to map the value of the AQI_Category to respective integers.

In [24]:
# Label Encoding for ordinal categories (AQI_Category has natural order)
print("\nLabel Encoding for AQI_Category (ordinal)...")
aqi_order = ['Good', 'Moderate', 'Unhealthy for Sensitive', 'Unhealthy', 'Very Unhealthy']
# Fill here

aqi_mapping = {
    'Good': 0,
    'Moderate': 1,
    'Unhealthy for Sensitive': 2,
    'Unhealthy': 3,
    'Very Unhealthy': 4
}
df_processed['AQI_Category_Encoded'] = df_processed['AQI_Category'].map(aqi_mapping)

print(f"  Mapping: {aqi_mapping}")
df_processed.head()


Label Encoding for AQI_Category (ordinal)...
  Mapping: {'Good': 0, 'Moderate': 1, 'Unhealthy for Sensitive': 2, 'Unhealthy': 3, 'Very Unhealthy': 4}


,DateTime,StationID,PM2.5,PM10,NO2,SO2,O3,CO,Temperature,Humidity,...,AQI_Category,Hour,Year,Month,Day,DayOfWeek,IsWeekend,TimePeriod,PM2.5_Level,AQI_Category_Encoded
0,2024-01-01 00:00:00,STATION_C,39.967142,73.428050,40.556577,17.385882,56.370175,1.258677,16.872084,65.781983,...,Unhealthy for Sensitive,0,2024,1,1,0,0,Night,Sensitive,2
1,2024-01-01 01:00:00,STATION_C,38.793738,62.491596,40.556577,22.743510,56.370175,0.982323,15.924396,51.618483,...,Unhealthy for Sensitive,1,2024,1,1,0,0,Night,Sensitive,2
2,2024-01-01 02:00:00,STATION_A,51.476885,97.711619,50.132296,27.958366,37.279782,1.039817,19.935899,62.911165,...,Unhealthy for Sensitive,2,2024,1,1,0,0,Night,Sensitive,2
3,2024-01-01 03:00:00,STATION_D,64.372434,92.734039,50.132296,17.852464,60.408011,1.047789,20.751439,42.309779,...,Unhealthy,3,2024,1,1,0,0,Night,Unhealthy,3
4,2024-01-01 04:00:00,STATION_C,38.793738,101.462011,57.659714,24.924136,36.624919,0.950957,17.376939,78.646998,...,Unhealthy for Sensitive,4,2024,1,1,0,0,Night,Sensitive,2


In [25]:
df_processed.columns

Index(['DateTime', 'StationID', 'PM2.5', 'PM10', 'NO2', 'SO2', 'O3', 'CO',
       'Temperature', 'Humidity', 'WindSpeed', 'Weather', 'AQI',
       'AQI_Category', 'Hour', 'Year', 'Month', 'Day', 'DayOfWeek',
       'IsWeekend', 'TimePeriod', 'PM2.5_Level', 'AQI_Category_Encoded'],
      dtype='str')

In [26]:
df_processed.info()

<class 'pandas.DataFrame'>
Index: 2002 entries, 0 to 2009
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   DateTime              2002 non-null   datetime64[us]
 1   StationID             2002 non-null   str           
 2   PM2.5                 2002 non-null   float64       
 3   PM10                  2002 non-null   float64       
 4   NO2                   2002 non-null   float64       
 5   SO2                   2002 non-null   float64       
 6   O3                    2002 non-null   float64       
 7   CO                    2002 non-null   float64       
 8   Temperature           2002 non-null   float64       
 9   Humidity              2002 non-null   float64       
 10  WindSpeed             2002 non-null   float64       
 11  Weather               2002 non-null   str           
 12  AQI                   2002 non-null   float64       
 13  AQI_Category          2002 non-nul

Extra Note:
PM2.5_Level showing 1982 non-null values, which happen due to the existance of minimal value in "PM2.5". There are 20 rows with negative values.
